In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import warnings; warnings.filterwarnings('ignore')
from datetime import datetime

sns.set_theme(style='whitegrid')
SEED = 42

In [ ]:
#chargement des données et affichage des premières lignes
df = pd.read_excel('../data/Dataset_complet_Meteo.xlsx')
df.head()

In [ ]:
# Reconstruction des colonnes corrompues
def reconstruct(val):
    if isinstance(val, datetime):
        return float(f"{val.day}.{val.month}")
    return val

cols_corrompues = [
    "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
    "apparent_temperature_max", "apparent_temperature_min", "apparent_temperature_mean",
    "wind_speed_10m_max", "wind_gusts_10m_max",
    "shortwave_radiation_sum", "et0_fao_evapotranspiration",
    "precipitation_sum", "rain_sum", "longitude", "latitude"
]

for col in cols_corrompues:
    df[col] = df[col].apply(reconstruct)
    df[col] = pd.to_numeric(df[col], errors='coerce')

df["sunshine_duration"]= pd.to_numeric(df["sunshine_duration"], errors='coerce')


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# ─────────────────────────────────────────────
# HYPOTHÈSE 1
# Les températures maximales varient significativement selon les régions
# ─────────────────────────────────────────────
fig1 = px.box(
    df,
    x='region',
    y='temperature_2m_max',
    color='region',
    title="Distribution des températures maximales par région",
    labels={'temperature_2m_max': 'Température max (°C)', 'region': 'Région'},
    template='plotly_white'
)
fig1.update_layout(showlegend=False, xaxis_tickangle=-45)
fig1.show()

In [ ]:
# ─────────────────────────────────────────────
# HYPOTHÈSE 2
# Les précipitations suivent une saisonnalité marquée au cours de l'année
# ─────────────────────────────────────────────
df['mois'] = df['time'].dt.month_name().str[:3]
precip_mois = df.groupby('mois')['precipitation_sum'].mean().reset_index()

fig2 = px.bar(
    precip_mois,
    x='mois',
    y='precipitation_sum',
    title="Précipitations moyennes par mois (toutes villes confondues)",
    labels={'precipitation_sum': 'Précipitation moyenne (mm)', 'mois': 'Mois'},
    template='plotly_white',
    color='precipitation_sum',
    color_continuous_scale='Blues'
)
fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

In [ ]:
# ─────────────────────────────────────────────
# HYPOTHÈSE 3
# Il existe une forte corrélation entre température et rayonnement solaire
# ─────────────────────────────────────────────
fig3 = px.scatter(
    df.sample(3000, random_state=42),  # échantillon pour la lisibilité
    x='shortwave_radiation_sum',
    y='temperature_2m_max',
    color='region',
    opacity=0.5,
    title="Corrélation entre rayonnement solaire et température maximale",
    labels={
        'shortwave_radiation_sum': 'Rayonnement solaire (MJ/m²)',
        'temperature_2m_max': 'Température max (°C)'
    },
    template='plotly_white',
    trendline='ols'
)
fig3.show()

df['month'] = df['time'].dt.month_name().str[:3]
monthly = df.groupby('month').agg(
    temp_moy=('temperature_2m_mean', 'mean'),
    pluie_moy=('precipitation_sum', 'mean')
)

fig3, ax1 = plt.subplots(figsize=(11, 4))
mois = ['Jan','Fév','Mar','Avr','Mai','Juin','Juil','Aoû','Sep','Oct','Nov','Déc']
ax1.bar(mois, monthly['pluie_moy'], color='steelblue', alpha=0.6, label='Précipitations (mm)')
ax2 = ax1.twinx()
ax2.plot(mois, monthly['temp_moy'], color='tomato', marker='o', lw=2, label='Température (°C)')
ax1.set_title('Saisonnalité — Température & Précipitations', fontweight='bold')
ax1.set_ylabel('Précipitations (mm)'); ax2.set_ylabel('Température (°C)')
plt.tight_layout(); plt.show()

df['annee'] = df['time'].dt.year
temp_annee = df.groupby('annee')['temperature_2m_mean'].mean().reset_index()

fig5,ax1 = px.line(
    temp_annee,
    x='annee',
    y='temperature_2m_mean',
    markers=True,
    title="Évolution de la température moyenne annuelle (2020–2025)",
    labels={'temperature_2m_mean': 'Température moyenne (°C)', 'annee': 'Année'},
    template='plotly_white'
)
fig5.show()

In [ ]:
# ─────────────────────────────────────────────
# HYPOTHÈSE 4
# La vitesse du vent est plus élevée dans certaines régions que d'autres
# ─────────────────────────────────────────────
vent_region = df.groupby('region')['wind_speed_10m_max'].mean().reset_index()

fig4 = px.bar(
    vent_region.sort_values('wind_speed_10m_max', ascending=False),
    x='region',
    y='wind_speed_10m_max',
    color='wind_speed_10m_max',
    color_continuous_scale='Teal',
    title="Vitesse moyenne du vent maximum par région",
    labels={'wind_speed_10m_max': 'Vitesse vent max moy. (km/h)', 'region': 'Région'},
    template='plotly_white'
)
fig4.update_layout(xaxis_tickangle=-45)
fig4.show()

In [ ]:
# ─────────────────────────────────────────────
# HYPOTHÈSE 5
# La température moyenne a évolué (tendance) entre 2020 et 2025
# ─────────────────────────────────────────────
df['annee'] = df['time'].dt.year
temp_annee = df.groupby('annee')['temperature_2m_mean'].mean().reset_index()

fig5 = px.line(
    temp_annee,
    x='annee',
    y='temperature_2m_mean',
    markers=True,
    title="Évolution de la température moyenne annuelle (2020–2025)",
    labels={'temperature_2m_mean': 'Température moyenne (°C)', 'annee': 'Année'},
    template='plotly_white'
)
fig5.show()

In [ ]:
# ─────────────────────────────────────────────
# HYPOTHÈSE 6
# L'évapotranspiration est liée à la température et au rayonnement solaire
# ─────────────────────────────────────────────
fig6 = px.scatter(
    df.sample(200, random_state=42),
    x='temperature_2m_mean',
    y='et0_fao_evapotranspiration',
    color='shortwave_radiation_sum',
    color_continuous_scale='Oranges',
    opacity=0.6,
    title="Évapotranspiration en fonction de la température et du rayonnement",
    labels={
        'temperature_2m_mean': 'Température moyenne (°C)',
        'et0_fao_evapotranspiration': 'Évapotranspiration (mm)',
        'shortwave_radiation_sum': 'Rayonnement (MJ/m²)'
    },
    template='plotly_white'
)
fig6.show()

In [ ]:
# ─────────────────────────────────────────────
# HYPOTHÈSE 7
# Certaines villes reçoivent beaucoup plus de pluie que d'autres
# ─────────────────────────────────────────────
pluie_ville = df.groupby('city')['rain_sum'].mean().reset_index()
pluie_ville = pluie_ville.sort_values('rain_sum', ascending=False).head(20)

fig7 = px.bar(
    pluie_ville,
    x='city',
    y='rain_sum',
    color='rain_sum',
    color_continuous_scale='Blues',
    title="Top 20 villes avec le plus de pluie moyenne journalière",
    labels={'rain_sum': 'Pluie moyenne (mm)', 'city': 'Ville'},
    template='plotly_white'
)
fig7.update_layout(xaxis_tickangle=-45)
fig7.show()

In [ ]:
df.sunshine_duration.unique

In [ ]:
# ─────────────────────────────────────────────
# HYPOTHÈSE 8
# La durée d'ensoleillement influence directement la température ressentie
# ─────────────────────────────────────────────
fig8 = px.scatter(
    df.sample(200, random_state=42),
    x='sunshine_duration',
    y='apparent_temperature_max',
    color='region',
    opacity=0.5,
    title="Ensoleillement vs température ressentie maximale",
    labels={
        'sunshine_duration': 'Durée ensoleillement (s)',
        'apparent_temperature_max': 'Température ressentie max (°C)'
    },
    template='plotly_white',
    trendline='ols'
)
fig8.show()

In [ ]:
# ─────────────────────────────────────────────
# HYPOTHÈSE 9
# La matrice de corrélation révèle des liens forts entre variables climatiques
# ─────────────────────────────────────────────
vars_corr = [
    'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
    'apparent_temperature_mean', 'precipitation_sum', 'rain_sum',
    'wind_speed_10m_max', 'shortwave_radiation_sum',
    'et0_fao_evapotranspiration'
]

corr_matrix = df[vars_corr].corr().round(2)

fig9 = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title="Matrice de corrélation des variables climatiques",
    template='plotly_white'
)
fig9.update_layout(width=800, height=700)
fig9.show()

In [ ]:
# ─────────────────────────────────────────────
# HYPOTHÈSE 10
# La répartition géographique des températures varie selon la latitude/longitude
# ─────────────────────────────────────────────
temp_ville = df.groupby(['city', 'latitude', 'longitude'])['temperature_2m_mean'].mean().reset_index()

fig10 = px.scatter_mapbox(
    temp_ville,
    lat='latitude',
    lon='longitude',
    color='temperature_2m_mean',
    size='temperature_2m_mean',
    hover_name='city',
    color_continuous_scale='RdYlGn_r',
    zoom=5,
    title="Température moyenne par ville (carte géographique)",
    labels={'temperature_2m_mean': 'Température moy. (°C)'},
    mapbox_style='open-street-map'
)
fig10.show()